# FoML lab starter — 20 Aug 2026

Catching your **first attended FoML lab**. Two layers:

1. **Week-1 announcement (6 Aug, Aniket / Kurhekar):** Python on [Google Colab](https://colab.research.google.com/), then arrays. Tutorial: [CS231n Python–NumPy](https://cs231n.github.io/python-numpy-tutorial/).
2. **Today (20 Aug, group chat):** combined lab 14:00–16:00. Experiments are **Decision Trees + Linear Regression**. **Evaluation is next week.**

Run cells with **Shift+Enter**. Colab already has `numpy`, `matplotlib`, `sklearn`.


## 0. Colab mechanics (do this first in the lab)
- `Runtime → Run all` if TA wants a full demo.
- `File → Save a copy in Drive` so you do not lose work if the runtime disconnects.
- Cells share one kernel: a variable defined above is visible below.
- If a cell errors, fix it and re-run **that cell and everything below it**.


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

print("Python", sys.version.split()[0])
print("numpy", np.__version__)


## 1. Python warmup (what the 6 Aug lab asked)

Interpreted = you run **one cell**, see output, then write the next. Same idea as REPL / Jupyter.


In [ ]:
# numbers / strings / bools
x = 3
print(x ** 2, type(x))
t, f = True, False
print(t and f, t or f, not t)

# list = Python "array" (resizable, mixed types). Index from 0. Slice [start:end).
xs = [3, 1, 2]
print(xs[0], xs[-1], xs[1:3])
squares = [n ** 2 for n in range(5) if n % 2 == 0]
print("even squares", squares)

# dict = key → value
d = {"cat": "cute", "dog": "furry"}
print(d["cat"], "fish" in d)

def sign(n):
    if n > 0:
        return "positive"
    if n < 0:
        return "negative"
    return "zero"

print([sign(v) for v in [-1, 0, 2]])


## 2. NumPy arrays (this is the FoML data type)

A **list of lists** is fine for tiny demos. For ML you want `np.ndarray`: one type, fast, vectorised.

| idea | meaning |
|------|---------|
| `shape` | sizes along each axis, e.g. `(3, 4)` = 3 rows, 4 cols |
| `a[i, j]` | row i, column j |
| `a[:2, 1:3]` | first 2 rows, columns 1 and 2 (end exclusive) |
| `*` | **elementwise** multiply |
| `@` or `np.dot` | matrix / inner product |
| `a[a > 2]` | boolean mask |


In [ ]:
a = np.array([1, 2, 3])
b = np.array([[1, 2, 3], [4, 5, 6]])
print("a", a, "shape", a.shape)
print("b\\n", b, "shape", b.shape)
print("b[0, 2] =", b[0, 2])

print("zeros\\n", np.zeros((2, 3)))
print("identity\\n", np.eye(3))

# slicing is a *view* — changing the slice can change the original
c = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
s = c[:2, 1:3]
s[0, 0] = 99
print("after slice write\\n", c)

x = np.array([[1., 2.], [3., 4.]])
y = np.array([[5., 6.], [7., 8.]])
print("elementwise *\\n", x * y)
print("matrix @\\n", x @ y)
print("col sums", np.sum(x, axis=0), "row sums", np.sum(x, axis=1))

# broadcasting: add a vector to every row
M = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9], [10, 11, 12]])
v = np.array([1, 0, 1])
print("M + v\\n", M + v)
print("mask M > 5 →", M[M > 5])


## 3. Linear regression (today)

Model: \( y = a_0 + a_1 x \). Fit by **least squares** (minimise sum of squared residuals), which is what Kurhekar’s regression notes do.

Viva line: sklearn’s `LinearRegression` solves that same normal equation; `coef_` is \(a_1\), `intercept_` is \(a_0\).


In [ ]:
# toy data: hours studied vs score
x = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
y = np.array([1.5, 3.2, 4.1, 5.8, 6.9, 8.4, 9.1, 11.0], dtype=float)

# --- numpy least squares: design matrix [1, x] ---
X_np = np.column_stack([np.ones_like(x), x])
a0, a1 = np.linalg.lstsq(X_np, y, rcond=None)[0]
print(f"numpy  y = {a0:.3f} + {a1:.3f} x")

# --- sklearn (what TAs usually want to see) ---
X = x.reshape(-1, 1)          # sklearn wants 2-D: (n_samples, n_features)
lin = LinearRegression().fit(X, y)
print(f"sklearn y = {lin.intercept_:.3f} + {lin.coef_[0]:.3f} x")
y_hat = lin.predict(X)
print("MSE", mean_squared_error(y, y_hat), "R^2", r2_score(y, y_hat))

plt.figure(figsize=(5, 3.5))
plt.scatter(x, y, label="data")
xs = np.linspace(x.min(), x.max(), 50)
plt.plot(xs, lin.predict(xs.reshape(-1, 1)), label="fit")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title("Linear regression")
plt.show()


## 4. Decision tree (today)

Kurhekar’s play/go-out example. Split by **information gain / entropy** (not Gini, unless they ask).

Root should come out **Weather**: Cloudy → Yes; Sunny → Humidity; Rainy → Wind.


In [ ]:
rows = [
    # weather, temperature, humidity, wind, play
    ["Sunny",  "Hot",  "High",   "Weak",   "No"],
    ["Cloudy", "Hot",  "High",   "Weak",   "Yes"],
    ["Sunny",  "Mild", "Normal", "Strong", "Yes"],
    ["Cloudy", "Mild", "High",   "Strong", "Yes"],
    ["Rainy",  "Mild", "High",   "Strong", "No"],
    ["Rainy",  "Cool", "Normal", "Strong", "No"],
    ["Rainy",  "Mild", "High",   "Weak",   "Yes"],
    ["Sunny",  "Hot",  "High",   "Strong", "No"],
    ["Cloudy", "Hot",  "Normal", "Weak",   "Yes"],
    ["Rainy",  "Mild", "High",   "Strong", "No"],
]
cols = ["weather", "temp", "humidity", "wind"]
X_cat = np.array([r[:4] for r in rows])
y_cat = np.array([r[4] for r in rows])

encoders = {c: LabelEncoder().fit(X_cat[:, i]) for i, c in enumerate(cols)}
X_num = np.column_stack([encoders[c].transform(X_cat[:, i]) for i, c in enumerate(cols)])
y_enc = LabelEncoder().fit(y_cat)
y_num = y_enc.transform(y_cat)

tree = DecisionTreeClassifier(criterion="entropy", random_state=0)
tree.fit(X_num, y_num)
pred = tree.predict(X_num)
print("accuracy (train, tiny n=10)", accuracy_score(y_num, pred))
print("confusion matrix\\n", confusion_matrix(y_num, pred, labels=y_enc.transform(["No", "Yes"])))
print("classes", list(y_enc.classes_))
for c in cols:
    print(c, "→", list(encoders[c].classes_))

plt.figure(figsize=(10, 6))
plot_tree(tree, feature_names=cols, class_names=list(y_enc.classes_), filled=True)
plt.title("Decision tree (entropy)")
plt.show()


## 5. If they give a CSV / sklearn dataset instead

Same pattern: load → `reshape` if 1 feature → `fit` → `predict` → metric + plot.


In [ ]:
# Uncomment if they ask for a built-in dataset:
# from sklearn.datasets import load_iris
# data = load_iris()
# tree2 = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=0)
# tree2.fit(data.data, data.target)
# print(accuracy_score(data.target, tree2.predict(data.data)))
print("ready. File → Download .ipynb before you leave the lab.")


## Viva, 30 seconds
- **Colab** = Jupyter in the browser; Google runs the Python, not your laptop.
- **NumPy array** vs list: one dtype, `shape`, vectorised ops; `*` is not matrix multiply.
- **Linreg:** minimise \(\sum (y_i - a_0 - a_1 x_i)^2\). `intercept_` / `coef_`.
- **Tree:** pick the split with highest information gain (entropy drop). Overfitting = deep tree that memorizes the 10 rows — that is why evaluation will want train vs test later.
